# Medidas de heterogeneidad

Práctica 4 · ENDIREH 2021

**Estado:** configuración inicial; análisis pendiente.

**Objetivo para la siguiente etapa:** Calcular IQV, Gini-Simpson y entropía de Shannon sobre variables categóricas.

La entrada es el CSV renombrado, generado desde el archivo limpio de la Práctica 3 en `ajuste_columnas_practica_4.ipynb`. Por ahora solo se preparan imports, carga y revisión de columnas.


## 1. Configuración del proyecto

Selecciona el entorno Python `endireh` antes de ejecutar las celdas.


In [5]:
from pathlib import Path
import sys

# Funciona si Jupyter se inicia en la raíz o dentro de notebooks/.
RUTA_PROYECTO = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "config" / "rutas.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError(
        "No se encontró config/rutas.py. Abre el notebook dentro del proyecto."
    )
if str(RUTA_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RUTA_PROYECTO))


In [6]:
import polars as pl
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from config.rutas import RUTA_ENDIREH_RENOMBRADO_CSV

sns.set_theme(style="whitegrid")
pl.Config.set_tbl_rows(100)


polars.config.Config

## 2. Carga del dataset procesado

Se conserva el CSV original. La validación de columnas no garantiza que los datos estén listos para todos los cálculos.


In [7]:
if not RUTA_ENDIREH_RENOMBRADO_CSV.is_file():
    raise FileNotFoundError(
        f"No se encontró el CSV procesado: {RUTA_ENDIREH_RENOMBRADO_CSV}. "
        "Ejecuta primero notebooks/ajuste_columnas_practica_4.ipynb."
    )

df = pl.read_csv(RUTA_ENDIREH_RENOMBRADO_CSV)

# Solo se valida la estructura básica; no se limpian ni transforman datos.
columnas_requeridas = {
    "cve_entidad", "nom_entidad", "sufrio_violencia_pareja", "factor_expansion"
}
columnas_faltantes = columnas_requeridas.difference(df.columns)
if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas: {sorted(columnas_faltantes)}")

print(f"Archivo: {RUTA_ENDIREH_RENOMBRADO_CSV.name}")
print(f"Filas: {df.height:,} | Columnas: {df.width}")
df.head(5)


Archivo: endireh_2021_renombrado.csv
Filas: 110,127 | Columnas: 24


cve_entidad,nom_entidad,cve_municipio,nom_municipio,anios_sin_convivencia_pareja,tipo_union,tipo_cuestionario,estado_civil_id,estado_civil_desc,estrato_sociodemografico_muestreo,entrevistada_trabaja_id,entrevistada_trabaja_desc,ingreso_laboral_entrevistada,dinero_propio_id,dinero_propio_desc,apoyo_gobierno_id,apoyo_gobierno_desc,hogar_tiene_ahorros_id,hogar_tiene_ahorros_desc,vivienda_propiedad_miembro_hogar_id,vivienda_propiedad_miembro_hogar_desc,sufrio_violencia_pareja,factor_expansion,anio_encuesta
i64,str,i64,str,i64,i64,str,i64,str,i64,i64,str,f64,i64,str,i64,str,i64,str,i64,str,i64,i64,i64
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,3,"""A1""",5,"""Casada""",4,1,"""Sí""",10000.0,1,"""Sí""",2,"""No""",2,"""No""",1,"""Sí""",0,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,3,"""A1""",5,"""Casada""",4,2,"""No""",null,2,"""No""",1,"""Sí""",2,"""No""",1,"""Sí""",1,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",13,3,"""B2""",4,"""Viuda""",4,2,"""No""",null,2,"""No""",1,"""Sí""",2,"""No""",2,"""No""",0,227,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,null,"""C1""",6,"""Soltera""",4,1,"""Sí""",8500.0,1,"""Sí""",2,"""No""",1,"""Sí""",1,"""Sí""",0,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",20,3,"""B1""",2,"""Separada""",2,1,"""Sí""",400.0,1,"""Sí""",2,"""No""",2,"""No""",1,"""Sí""",1,155,2021


## 3. Variables y ponderación

Se seleccionaron variables cualitativas relacionadas con características sociales y económicas de las mujeres encuestadas: estado civil, disponibilidad de dinero propio, recepción de apoyo gubernamental y presencia de ahorros en el hogar. Sobre estas variables se analizará qué tan equilibrada o concentrada se encuentra la distribución de sus categorías.
 y documentar el uso del factor de expansión.

Consulta las correspondencias y las decisiones heredadas en `ajuste_columnas_practica_4.ipynb` antes de elegir variables.


In [12]:
variables_cualitativas = [
    "estado_civil_desc",
    "dinero_propio_desc",
    "apoyo_gobierno_desc",
    "hogar_tiene_ahorros_desc"
]
for variable in variables_cualitativas:

    print(f"\n--- {variable} ---")

    frecuencias = (
        df
        .group_by(variable)
        .len()
        .rename({"len": "frecuencia"})
        .with_columns(
            (
                pl.col("frecuencia")
                / pl.col("frecuencia").sum()
            ).alias("proporcion")
        )
        .sort("frecuencia", descending=True)
    )

    print(frecuencias)


--- estado_civil_desc ---
shape: (6, 3)
┌───────────────────┬────────────┬────────────┐
│ estado_civil_desc ┆ frecuencia ┆ proporcion │
│ ---               ┆ ---        ┆ ---        │
│ str               ┆ u32        ┆ f64        │
╞═══════════════════╪════════════╪════════════╡
│ Casada            ┆ 44977      ┆ 0.40841    │
│ Unión libre       ┆ 23597      ┆ 0.214271   │
│ Soltera           ┆ 20002      ┆ 0.181627   │
│ Viuda             ┆ 9750       ┆ 0.088534   │
│ Separada          ┆ 8628       ┆ 0.078346   │
│ Divorciada        ┆ 3173       ┆ 0.028812   │
└───────────────────┴────────────┴────────────┘

--- dinero_propio_desc ---
shape: (2, 3)
┌────────────────────┬────────────┬────────────┐
│ dinero_propio_desc ┆ frecuencia ┆ proporcion │
│ ---                ┆ ---        ┆ ---        │
│ str                ┆ u32        ┆ f64        │
╞════════════════════╪════════════╪════════════╡
│ Sí                 ┆ 57662      ┆ 0.523595   │
│ No                 ┆ 52465      ┆ 0.476405   

## 4. Cálculos

Pendiente: implementar las medidas en funciones reutilizables de `src/statistics/` y utilizarlas aquí.


## 5. Tablas y visualizaciones

Pendiente: generar las tablas y gráficas correspondientes al análisis.


## 6. Interpretación y limitaciones

Pendiente: explicar los resultados en el contexto de violencia contra las mujeres, las limitaciones de los datos y las decisiones tomadas.
